# 02. Data Validation & Leakage Verification
Runs `CorpusValidator` (content-quality checks) and `DataLeakageChecker` (zero-overlap audit) against the master corpus and its splits.

In [1]:
# ============================================================
# PATH BOOSTER -- guarantees project root in sys.path & CWD.
# Matches notebooks/00_setup_environment.ipynb's convention: this repo
# is deployed both to Colab (fresh `git clone`, folder "Ekegusii-LLM-Translation")
# and to Kineses Cloud / similar Jupyter hosts (pre-placed at
# ~/Ekegusii-LLM-Translation-main -- the "-main" suffix comes from GitHub's
# "Download ZIP" naming). Do not assume either folder name is the cwd.
# ============================================================
import os
import sys

REPO_NAME = "Ekegusii-LLM-Translation"


def _find_project_root():
    if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
        if not os.path.exists(REPO_NAME):
            os.system(f"git clone https://github.com/aykahsay/{REPO_NAME}.git")
            os.system(f"pip install -q -r {REPO_NAME}/requirements.txt")
        return os.path.abspath(REPO_NAME)

    try:
        cwd = os.getcwd()
    except FileNotFoundError:
        cwd = os.path.expanduser("~")
        os.chdir(cwd)

    home = os.path.expanduser("~")
    for candidate in (f"{REPO_NAME}-main", REPO_NAME):
        proj_dir = os.path.join(home, candidate)
        if os.path.isdir(proj_dir):
            return proj_dir

    if os.path.exists("src") and os.path.exists("data"):
        return cwd
    if os.path.basename(cwd) == "notebooks" and os.path.exists(os.path.join("..", "src")):
        return os.path.abspath("..")

    return cwd


project_root = _find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Project root: {project_root}")


Project root: C:\Users\Admin\OneDrive - United States International University (USIU)\Documents\NLP\Multilogual_transaltion_nlp


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.validator import CorpusValidator
from src.master_corpus.integrity import DataLeakageChecker
from src.utils.constants import SUPPORTED_LANGUAGES

manager = MasterCorpusManager()
validator = CorpusValidator()

## Content validation

In [4]:
sentence_df = manager.load_sentence_corpus()
report = validator.validate(
    sentence_df, 'master_sentence_corpus', 'concept_id', list(SUPPORTED_LANGUAGES),
    max_null_rate=0.30,
)
print(f'Valid: {report.is_valid}')
print(f'Issues: {report.issues}')

INFO | Loaded Master Sentence Corpus: 49,277 concepts.


INFO | [master_sentence_corpus] Validation passed (49,277 rows).


Valid: True
Issues: []


In [5]:
lexical_df = manager.load_lexical_corpus()
lex_report = validator.validate(
    lexical_df, 'master_lexical_corpus', 'lexicon_id', list(SUPPORTED_LANGUAGES),
    max_null_rate=1.0,  # English is 100% empty in this corpus -- see docs/datasets.md
)
print(f'Issues: {lex_report.issues}')

INFO | Loaded Master Lexical Corpus: 268 terms.


INFO | [master_lexical_corpus] Validation passed (268 rows).


Issues: []


## Zero-leakage audit
Every experiment depends on this passing.

In [6]:
checker = DataLeakageChecker(manager)
passed = checker.verify_all()
print(f'Leakage audit passed: {passed}')

INFO | === Starting Master Corpus Data Leakage Audit ===


INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


INFO | Loaded dataset split [master_val.csv]: 4,928 rows.


INFO | Loaded dataset split [master_test.csv]: 4,928 rows.


INFO | 1/2 [ID Audit] Zero Concept ID overlap confirmed across splits.


WARNING | Notice: 746 identical English sentences between Train & Test.


INFO | 2/2 [Text Audit] Sentence text overlap within acceptable limits.


INFO | ✅ [PASSED] 0% Data Leakage Audit Verified Successfully!


Leakage audit passed: True
